In [2]:
import { initChatModel } from "langchain/chat_models/universal";
import { SystemMessage, HumanMessage } from "@langchain/core/messages";
import { ChatOpenAI } from "@langchain/openai";
import { tool } from "@langchain/core/tools";
import { z } from "zod";


const llm = new ChatOpenAI({ model: "gpt-4o-mini" });

const response1 = await llm.invoke("Hola, como estas?");
console.log("response1:", response1.text);



response1: ¡Hola! Estoy aquí y listo para ayudarte. ¿En qué puedo asistirte hoy?


In [3]:
const response2 = await llm.invoke("Que clima hace en la ciudad de Bogota?");
console.log("response2:", response2.text);

response2: El clima en Bogotá es considerado un clima de montaña, caracterizado por ser fresco y templado durante todo el año. La ciudad se encuentra a una altitud de aproximadamente 2,640 metros sobre el nivel del mar, lo que influye en sus condiciones climáticas. 

Las temperaturas en Bogotá oscilan generalmente entre los 10 °C y 20 °C. Durante el día, las temperaturas pueden llegar a ser más cálidas, especialmente en las horas cercanas al mediodía, pero suelen descender por la tarde y la noche. Además, Bogotá experimenta lluvias a lo largo del año, con dos temporadas de lluvias principales: de abril a mayo y de octubre a noviembre. 

Es recomendable llevar ropa adecuada para el clima fresco y estar preparado para la posibilidad de lluvias en cualquier momento.


In [4]:
const systemPrompt = `
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan.

Y tus productos son:
- Computadora
- Mouse
- Teclado
- Audifonos
- Mousepad
`;

const messages = [
  new SystemMessage(systemPrompt),
  new HumanMessage("Dime los productos que ofreces en la tienda"),
];

const response3 = await llm.invoke(messages);
console.log("response3:", response3.text);

response3: En la tienda ofrecemos los siguientes productos:

- Computadora
- Mouse
- Teclado
- Audífonos
- Mousepad

Si necesitas más información sobre alguno de ellos o ayuda para elegir, ¡hazmelo saber!


In [5]:
const getProducts = tool(
  async () => {
    const response = await fetch("https://api.escuelajs.co/api/v1/products");
    const products: { title: string; price: number }[] = await response.json();
    return products
      .map((product) => `${product.title} - ${product.price}`)
      .join("\n");
  },
  {
    name: "get_products",
    description: "Get the products that the store offers filter by price",
    schema: z.object({}), // sin parámetros
  }
);

// ─── Invocar la tool directamente ────────────────────────────────────────────
const toolResult = await getProducts.invoke({});
console.log("toolResult:", toolResult);

toolResult: Chage title - 100
kilian - 56
gaesgaehaehadasg - 74
TestProduct - 456
New Product - 10
NogNOgeentest - 55
Laatsetsteatesga - 25
gahepgiapwehg - 565
New Product000w - 10
商品_7e6a5404-9383-4502-8a74-c33ed3afea22 - 10
商品_27184290-bff9-440d-8878-c1e6dd2c1a0d - 10
商品_ffe4ca5e-f7b7-4b5a-9584-284cb8d86290 - 10
商品_c92ffec6-d2f4-46b3-82b6-769e870f75ac - 10
商品_ffca8a58-8adb-4a27-ae8e-047891075044 - 10
商品_4d26a458-a947-4f32-9405-8769e7c5b57c - 10
商品_2ea6f27c-25e1-42a1-9480-a75fc2cf87a1 - 10
商品_87e092a4-1c19-4bbe-8071-20c631cc53de - 10
商品_7e30637c-237b-4294-81b0-67c3e7f088e6 - 10
商品_71252da2-35cd-4c76-9e8b-957b7d88f6fd - 10
商品_937bc3d9-94d1-4dfd-b19a-e1cbbe4f249f - 10
商品_12dd4d4d-1e28-4d0c-abbd-ade85134df72 - 10
商品_b3ad721f-a74d-44a4-a101-2e69828a5dbe - 10
商品_7a8bb658-d592-4f22-983b-6fe75ab7255e - 10
商品_edb1a090-eca7-40c0-a0df-69c58612eff3 - 10
商品_38927354-e3c3-493d-812f-e1512c5d88b9 - 10
商品_085ccd28-d7a7-40ae-872a-d8738c4cbeec - 10
商品_8fceba80-3f59-4498-9582-cd99521bbcd5 - 10
商品_a6338a

In [6]:
const getProducts = tool(
  async () => {
    const response = await fetch("https://api.escuelajs.co/api/v1/products");
    const products: { title: string; price: number }[] = await response.json();
    return products.map((p) => `${p.title} - ${p.price}`).join("\n");
  },
  {
    name: "get_products",
    description: "Get the products that the store offers filter by price",
    schema: z.object({}),
  }
);

// ─── Tool: get_weather ────────────────────────────────────────────────────────
const getWeather = tool(
  async ({ city }: { city: string }) => {
    // 1. Obtener coordenadas
    const geoRes = await fetch(
      `https://geocoding-api.open-meteo.com/v1/search?name=${city}&count=1`
    );
    const geoData = await geoRes.json();
    const { latitude, longitude } = geoData.results[0];

    // 2. Obtener clima
    const weatherRes = await fetch(
      `https://api.open-meteo.com/v1/forecast?latitude=${latitude}&longitude=${longitude}&current_weather=true`
    );
    const weatherData = await weatherRes.json();
    const { temperature, windspeed } = weatherData.current_weather;

    return `The weather in ${city} is ${temperature}C with ${windspeed}km/h of wind.`;
  },
  {
    name: "get_weather",
    description: "Get the weather of a city",
    schema: z.object({
      city: z.string().describe("The name of the city"),
    }),
  }
);

// ─── Invocar tool directamente ────────────────────────────────────────────────
const weatherResult = await getWeather.invoke({ city: "bogota" });
console.log("weatherResult:", weatherResult);

// ─── System prompt + bind tools ──────────────────────────────────────────────
const systemPrompt = `
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y dar el clima de la ciudad

Tus tools son:
- get_products: para obtener los productos que ofreces en la tienda
- get_weather: para obtener el clima de la ciudad
`;

const messages = [
  new SystemMessage(systemPrompt),
  new HumanMessage("Dime los productos que ofreces en la tienda"),
];

// ─── LLM con tools bindeadas ──────────────────────────────────────────────────
const llmWithTools = llm.bindTools([getProducts, getWeather]);

const response = await llmWithTools.invoke(messages);

// Equivalente a response.tool_calls
console.log("tool_calls:", response.tool_calls);

weatherResult: The weather in bogota is 17.7C with 10.9km/h of wind.
tool_calls: [
  {
    name: "get_products",
    args: {},
    type: "tool_call",
    id: "call_8W8yRrolkO4F7U5g85ERGo8L"
  }
]


In [7]:
const messages2 = [
  new SystemMessage(systemPrompt),
  new HumanMessage("Cual es el clima en la capital de Colombia ?"),
];

const response2 = await llmWithTools.invoke(messages2);
console.log("tool_calls:", response2.tool_calls);

tool_calls: [
  {
    name: "get_weather",
    args: { city: "Bogotá" },
    type: "tool_call",
    id: "call_fnxgrPBEBF9ZvL2tQ9DwYsGl"
  }
]
